# Falcao — National Labor-Justice Jurisprudence (CSJT)

`FalcaoScraper` queries the **Jurisprudencia Nacional** system of the Brazilian
Labor Justice (Justica do Trabalho), hosted by the CSJT at
<https://jurisprudencia.jt.jus.br>. A single search endpoint spans the **TST**
and all **24 TRTs** across five document collections.

| Method | Description |
|--------|-------------|
| `listar_decisoes` | Jurisprudence search — returns a `DataFrame` |
| `listar_decisoes_download` | Downloads the raw JSON pages to disk |
| `listar_decisoes_parse` | Parses a folder produced by `listar_decisoes_download` |

**Collections** (`colecao` argument): `acordaos` (default), `sentencas`,
`decisoesmonocraticas`, `precedentes`, `recursorevista`.

> Notes: the public (unauthenticated) endpoint only allows page sizes of **5**
> or **10** (`tamanho_pagina`) and caps results at 10,000 per query.

## Basic search

This example requests two pages. Always select a bounded number of pages for exploratory queries. The backend can respond with HTTP 429 and a long wait. Increasing `sleep_time` does not cancel an active block.

On 2026-10-03, five successful probes, including three consecutive calls in one session, each reported `x-rate-limit-remaining: 49`. Later that day, HTTP 429 reported a wait of 27,720 seconds at 19:17:44 UTC, pointing to release at 23:59:44 in Brasilia. This is consistent with the earlier midnight-reset hypothesis, but the quota size and the meaning of the successful-response counter remain unverified. The client does not use that counter to reject larger searches in advance. See [issue #350](https://github.com/jtrecenti/juscraper/issues/350).

If `listar_decisoes` fails after receiving pages, it preserves their JSON files in their own folder under the system temporary directory; the scraper's `download_path` is not used. The exception note gives the folder to pass to `listar_decisoes_parse` for a partial result. To continue an interrupted search instead, use `checkpoint_dir` (see *Resume an interrupted collection* below). Successful calls remove their temporary files.

A search of more than one page in the default relevance order emits a `UserWarning`, as the cell below does: that order changes between calls of the same search, so the pages of one collection may repeat or omit documents. Sort by date to avoid it (see *Resume an interrupted collection* below).

In [1]:
import juscraper as jus

falcao = jus.scraper("falcao")
df = falcao.listar_decisoes("dano moral", paginas=2, tamanho_pagina=10)
print(df.shape)
df.head(3)

Falcao/acordaos:   0%|          | 0/2 [00:00<?, ?it/s]

(20, 21)


,siglaClasseProcesso,classeProcesso,relator,tribunal,idGabinete,gabinete,idTurma,turma,textoAcordao,ementa,...,idDocumentoAcordao,referenciaLegislativa,prioridades,score,processo,colecao,data_julgamento,data_juntada,classe,classe_sigla
0,ROT,Recurso Ordinário Trabalhista,MARCO ANTONIO PAULINELLI DE CARVALHO,TRT3,118.0,Gabinete de Desembargador n. 27,21,11ª Turma,"<table width=""100%"" cellspacing=""2"" cellpaddin...",DANO MORAL COLETIVO. CARACTERIZAÇÃO. Comprovad...,...,52519235,"[art_186_ccb, art_794_clt, trt3, eca, art_319_...",None,163.56393,0010742-41.2019.5.03.0089,acordaos,08/07/2020,09/07/2020,Recurso Ordinário Trabalhista,ROT
1,AP,Agravo de Petição,Manoel Barbosa da Silva,TRT3,60.0,Gabinete de Desembargador n. 23,17,05ª Turma,"<table width=""100%"" cellspacing=""2"" cellpaddin...",None,...,54012484,"[trt3, prgf_2_art_791_clt, art_477_clt, iv_prg...",None,161.79614,0010172-33.2020.5.03.0182,acordaos,04/08/2020,07/08/2020,Agravo de Petição,AP
2,CSJT-PCA,Procedimento de Controle Administrativo,FRANCISCO JOSE PINHEIRO CRUZ,CSJT,NaN,None,405,Conselho Superior da Justiça do Trabalho,<html>\n <head>\n <style>\n <!--\n ...,PROCEDIMENTO DE CONTROLE ADMINISTRATIVO. TRT d...,...,e70315f87fe4b814d1066dd2ae86b328,"[art_24_resolucao_csjt_68_10, resolucao_cnj_83...",None,161.42017,0025151-21.2015.5.90.0000,acordaos,24/03/2017,30/03/2017,Procedimento de Controle Administrativo,CSJT-PCA


## Available columns

In [2]:
df.columns.tolist()

['siglaClasseProcesso',
 'classeProcesso',
 'relator',
 'tribunal',
 'idGabinete',
 'gabinete',
 'idTurma',
 'turma',
 'textoAcordao',
 'ementa',
 'possuiEmenta',
 'idDocumentoAcordao',
 'referenciaLegislativa',
 'prioridades',
 'score',
 'processo',
 'colecao',
 'data_julgamento',
 'data_juntada',
 'classe',
 'classe_sigla']

## Preview an ementa

The `acordaos` collection ships the ementa, returned as plain text (HTML stripped).

In [3]:
print(df["ementa"].iloc[0][:300])

DANO MORAL COLETIVO. CARACTERIZAÇÃO. Comprovado o ato ilícito praticado pela reclamada, com violação ao ordenamento jurídico e sonegação de direito trabalhista assegurado aos empregados (descanso semanal remunerado), é inegável a ofensa aos direitos fundamentais dos trabalhadores, sobretudo a dignid


## Using filters

Filters are validated by the `InputListarDecisoesFalcao` pydantic schema. Multi-value
filters accept a string or a list of strings.

In [4]:
df_filtrado = falcao.listar_decisoes(
    "hora extra",
    paginas=1,
    tribunais=["TST", "TRT3"],       # court(s)
    ordenacao="mais_recente",         # mais_relevante | mais_recente | menos_recente
    data_juntada_inicio="2023-01-01",  # ISO or DD/MM/YYYY, filters dataJuntada
    data_juntada_fim="2023-12-31",
)
print(df_filtrado.shape)
df_filtrado[["processo", "tribunal", "data_juntada"]].head()
df_filtrado.head()


Falcao/acordaos:   0%|          | 0/1 [00:00<?, ?it/s]

(10, 21)


,siglaClasseProcesso,classeProcesso,relator,tribunal,idGabinete,gabinete,idTurma,turma,textoAcordao,ementa,...,idDocumentoAcordao,referenciaLegislativa,prioridades,score,processo,colecao,data_julgamento,data_juntada,classe,classe_sigla
0,AIRR,Agravo de Instrumento em Recurso de Revista,SERGIO PINTO MARTINS,TST,None,None,22,8ª Turma,"<!DOCTYPE html><html xmlns=""http://www.w3.org/...",AGRAVO EM AGRAVO DE INSTRUMENTO EM RECURSO DE ...,...,fc34ba2ee76e19dd51779e85d17f274b,"[art_389_cpc, art_896_clt, sumula_126_tst, art...",None,NaN,0000152-95.2014.5.02.0351,acordaos,13/12/2023,26/12/2023,Agravo de Instrumento em Recurso de Revista,AIRR
1,RR,Recurso de Revista,SERGIO PINTO MARTINS,TST,None,None,22,8ª Turma,"<![CDATA[<p align=""justify""><b>A C Ó R D Ã O</...","<p align=""justify""><b>RECURSO DE REVISTA -REGÊ...",...,aa6472419237b828bfa9396814a1ecec,"[art_896_clt, art_7_cf, i_prgf_1_art_896_clt, ...",None,NaN,0000472-53.2022.5.09.0325,acordaos,13/12/2023,26/12/2023,Recurso de Revista,RR
2,AIRR,Agravo de Instrumento em Recurso de Revista,SERGIO PINTO MARTINS,TST,None,None,22,8ª Turma,"<!DOCTYPE html><html xmlns=""http://www.w3.org/...",AGRAVO DE INSTRUMENTO EM RECURSO DE REVISTA DO...,...,1fac777f2b238bdaffdc786c5b4347dd,"[v_sumula_331_tst, art_896_clt, i_prgf_1_art_8...",None,NaN,0000659-70.2020.5.06.0016,acordaos,13/12/2023,26/12/2023,Agravo de Instrumento em Recurso de Revista,AIRR
3,AIRR,Agravo de Instrumento em Recurso de Revista,SERGIO PINTO MARTINS,TST,None,None,22,8ª Turma,"<!DOCTYPE html><html xmlns=""http://www.w3.org/...",AGRAVO DE INSTRUMENTO EM RECURSO DE REVISTA IN...,...,66f00d3318ee717e73ad5619c740809e,"[i_prgf_1_art_896_clt, iii_prgf_1_art_896_clt,...",None,NaN,0000820-38.2021.5.09.0121,acordaos,13/12/2023,26/12/2023,Agravo de Instrumento em Recurso de Revista,AIRR
4,AIRR,Agravo de Instrumento em Recurso de Revista,SERGIO PINTO MARTINS,TST,None,None,22,8ª Turma,"<![CDATA[<p align=""justify"">A C Ó R D Ã O<br/>...","<p align=""justify""><b>7ª Turma</b><br/></p><p ...",...,e12d133d21853f9d128db32b62049d89,"[prgf_1_art_896_clt, sumula_126_tst, trt9, ii_...",None,NaN,0001084-91.2022.5.11.0015,acordaos,13/12/2023,26/12/2023,Agravo de Instrumento em Recurso de Revista,AIRR


## Querying different collections

The same search works across collections; each returns its own document shape,
normalized to a common core (`processo`, `colecao`, `tribunal`, ...).

Regional precedents can use `idTema` instead of `id`. Their `processo` key retains that identifier with a tribunal prefix; it is not a CNJ case number.

In [5]:
df_sent = falcao.listar_decisoes("acidente de trabalho", paginas=1, colecao="sentencas")
print("sentencas:", df_sent.shape)

df_sum = falcao.listar_decisoes("dano moral", paginas=1, colecao="precedentes")
print("precedentes:", df_sum.shape)
df_sum[["processo", "colecao", "tribunal"]].head()

Falcao/sentencas:   0%|          | 0/1 [00:00<?, ?it/s]

sentencas: (10, 21)


Falcao/precedentes:   0%|          | 0/1 [00:00<?, ?it/s]

precedentes: (10, 72)


,processo,colecao,tribunal
0,TST-SUM-392,precedentes,TST
1,TRT9-idTema43-IUJ_7_TRT9,precedentes,TRT9
2,TRT19-idIAC-4-13215-TRT19,precedentes,TRT19
3,TRT9-idTema88-IUJ_52_TRT9,precedentes,TRT9
4,TRT9-idTema64-IUJ_28_TRT9,precedentes,TRT9


## Download and parse separately

Each `listar_decisoes_download` call creates its own subfolder and saves one raw JSON file per page (named
`{colecao}_{page}.json`); `listar_decisoes_parse` reads a folder back into a `DataFrame`.

In [6]:
import tempfile, os, json

with tempfile.TemporaryDirectory() as tmp:
    pasta = falcao.listar_decisoes_download("dano moral", paginas=1, diretorio=tmp)
    arquivos = os.listdir(pasta)
    print("files:", arquivos)

    raw = json.load(open(os.path.join(pasta, arquivos[0])))
    print("quantidadeTotal:", raw["quantidadeTotal"])
    print("raw keys:", list(raw.keys()))

    df_parsed = falcao.listar_decisoes_parse(pasta)
    print("parsed:", df_parsed.shape)

Falcao/acordaos:   0%|          | 0/1 [00:00<?, ?it/s]

files: ['acordaos_0001.json']
quantidadeTotal: 10000
raw keys: ['documentos', 'temasTopFive', 'quantidadeTotal']
parsed: (10, 21)


## Resume an interrupted collection

The backend blocks the IP for hours after a small number of requests (HTTP 429, apparently a daily quota), so a large
search may not fit in one day. Pass `checkpoint_dir` to `listar_decisoes` or `listar_decisoes_download` to keep the
pages and a `manifest.json` in a directory of your choice, and repeat the call with `resume=True` to request only the
pages that are still missing:

```python
import requests

filtros = dict(colecao="acordaos", tribunais="TST", ordenacao="menos_recente",
               data_juntada_fim="2024-12-31")

try:
    df = falcao.listar_decisoes("dano moral", checkpoint_dir="coleta_dano_moral", **filtros)
except requests.HTTPError:
    pass  # quota exhausted: the pages received so far stay in the checkpoint

# later, once the IP is released
df = falcao.listar_decisoes("dano moral", checkpoint_dir="coleta_dano_moral", resume=True, **filtros)
```

- A new collection refuses an occupied `checkpoint_dir`; `resume=True` requires the checkpoint of the same search
  (term, collection, filters and `tamanho_pagina`) and fails before any request otherwise. `paginas` may change
  between sessions, and the order of the values in a list filter does not matter.
- Pages already on disk are never requested again, and a checkpoint that is already complete costs no request: the total recorded in the manifest stands.
- The returned `DataFrame` has every page in the checkpoint; `listar_decisoes_parse(checkpoint_dir)` reads it too.
- The index can change between sessions. If the total changes, the pages already saved are kept and a `UserWarning`
  says they may repeat or omit documents relative to the new ones; a change that keeps the total (always the case above the 10,000 cap) goes unnoticed. A closed `data_juntada_fim` in the past with
  `ordenacao="menos_recente"` keeps new documents from shifting the pages.
- The default relevance order changes between calls of the same search: measured live on 2026-11-16, page 3 of
  "dano moral" came back in three versions within three minutes, while `ordenacao="menos_recente"` with a closed
  `data_juntada_fim` returned the same page in three calls. A collection with `checkpoint_dir`, or of more than
  one page, in the default order emits a `UserWarning`, because pages requested at different moments may repeat or
  omit documents.
- `resume=True` fails when the directory has no checkpoint, so a wrong path never starts from scratch. In a daily
  script with a single call, pass `resume=(Path(checkpoint_dir) / "manifest.json").exists()`.
- There is no directory lock: two simultaneous runs on the same checkpoint request the same pages twice.

## TST coverage

The `tribunais="TST"` filter restricts any collection to the Superior Labor Court (TST). The cell below checks each collection with a broad term (`dano moral`): how many TST documents match, which courts the first page returns, and the oldest and newest dates when the first page is sorted each way (`menos_recente` / `mais_recente`). It sends up to 10 requests, which count toward the backend quota discussed above. The cell prints the day of the run.

How to read the table:

- `total_tst` is capped at 10,000 by the backend, so 10000 means 10,000 or more. To count beyond the cap, split the search into `data_juntada_inicio` / `data_juntada_fim` windows.
- `tribunais` should list only `TST`; anything else means the filter is not applied to that collection.
- A zero total means the collection holds no TST document for the term. The TST does not issue first-instance rulings, so `sentencas` is expected to be empty.
- `campo_data` names the date field used (`data_juntada` is the filing date). The dates bound the first page in each order, not the whole collection.
- TST precedents are súmulas and orientações jurisprudenciais, keyed as `TST-SUM-392` or `TST-OJ-SBDI2-130`; they are not decisions.

On 2026-11-15, the run below returned only `TST` in every collection with results. `acordaos` and `decisoesmonocraticas` reached the 10,000 cap, with first-page filing dates from 2016-01-08 to 2026-10-29 and from 2016-01-07 to 2026-09-24; `precedentes` held 32 TST documents, with first-page publication dates from 2012-09-27 to 2016-04-26. `sentencas` returned 0, as expected. `recursorevista` also returned 0 for the TST, which is a coverage gap rather than a filter failure: the same search returned only `TRT22` documents unfiltered and only `TRT2` documents with `tribunais="TRT2"`.

Falcao covers jurisprudence only. For TST case metadata and procedural movements, use the `datajud` aggregator, which maps `TST` to its public index.

In [7]:
import json
import tempfile
import time
from datetime import date
from pathlib import Path

import pandas as pd

falcao_tst = jus.scraper("falcao", verbose=0)  # no progress bar per request


def primeira_pagina_tst(colecao, ordenacao, diretorio):
    """First page (5 documents) of a TST-only search: raw total and parsed rows."""
    pasta = falcao_tst.listar_decisoes_download(
        "dano moral", paginas=1, tamanho_pagina=5, colecao=colecao,
        tribunais="TST", ordenacao=ordenacao, diretorio=diretorio,
    )
    bruto = json.loads(next(Path(pasta).glob("*.json")).read_text(encoding="utf-8"))
    time.sleep(falcao_tst.sleep_time)
    return bruto["quantidadeTotal"], falcao_tst.listar_decisoes_parse(pasta)


def limites_de_data(df):
    """First date field with values in ``df`` and its earliest and latest dates."""
    for campo in [c for c in ("data_juntada", "data_julgamento", "dataPublicacao") if c in df]:
        datas = pd.to_datetime(df[campo], format="%d/%m/%Y", errors="coerce").dropna()
        if len(datas):
            return {"campo_data": campo, "mais_antigo": datas.min(), "mais_recente": datas.max()}
    return {}


linhas = []
with tempfile.TemporaryDirectory() as tmp:
    for colecao in falcao_tst.COLECOES:
        total, antigos = primeira_pagina_tst(colecao, "menos_recente", tmp)
        linha = {"colecao": colecao, "total_tst": total}
        if total:
            # With 5 or fewer hits, the first page already holds every document.
            recentes = antigos if total <= 5 else primeira_pagina_tst(colecao, "mais_recente", tmp)[1]
            paginas = pd.concat([antigos, recentes])
            linha["tribunais"] = ", ".join(sorted(paginas["tribunal"].dropna().unique()))
            linha.update(limites_de_data(paginas))
        linhas.append(linha)

print("Checked on", date.today().isoformat())
pd.DataFrame(linhas)

Checked on 2026-11-15


,colecao,total_tst,tribunais,campo_data,mais_antigo,mais_recente
0,acordaos,10000,TST,data_juntada,2016-01-08,2026-10-29
1,sentencas,0,NaN,NaN,NaT,NaT
2,decisoesmonocraticas,10000,TST,data_juntada,2016-01-07,2026-09-24
3,precedentes,32,TST,dataPublicacao,2012-09-27,2016-04-26
4,recursorevista,0,NaN,NaN,NaT,NaT
